# Notebook 02: Feature Engineering

## Purpose
Compute Stage 1 (market structure) and Stage 2 (behavioral) features
for every market. These features feed directly into the ML models in NB 03.

## Inputs
- `data/raw/markets_polymarket.parquet`
- `data/raw/markets_kalshi.parquet`
- `data/raw/trades_polymarket.parquet`
- `data/raw/trades_kalshi.parquet`

## Outputs
- `data/processed/features_stage1.parquet` — one row per market
- `data/processed/features_stage2.parquet` — one row per market per time window

In [10]:
# Cell 2 — Imports
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

Path("../data/processed").mkdir(parents=True, exist_ok=True)

# Load raw data
pm_markets = pd.read_parquet("../data/raw/markets_polymarket.parquet")
kalshi_markets = pd.read_parquet("../data/raw/markets_kalshi.parquet")
pm_trades = pd.read_parquet("../data/raw/trades_polymarket.parquet")
kalshi_trades = pd.read_parquet("../data/raw/trades_kalshi.parquet")

print(f"Polymarket markets: {len(pm_markets):,}")
print(f"Kalshi markets:     {len(kalshi_markets):,}")
print(f"Polymarket trades:  {len(pm_trades):,}")
print(f"Kalshi trades:      {len(kalshi_trades):,}")

Polymarket markets: 100
Kalshi markets:     13
Polymarket trades:  346,500
Kalshi trades:      3,640


In [11]:
# Cell 3 — Clean and parse Polymarket trades
pm_trades["timestamp"] = pd.to_datetime(pm_trades["timestamp"], unit="s", utc=True)
pm_trades["size"] = pd.to_numeric(pm_trades["size"], errors="coerce")
pm_trades["price"] = pd.to_numeric(pm_trades["price"], errors="coerce")

# Parse market end dates
pm_markets["end_date"] = pd.to_datetime(pm_markets["end_date"], utc=True, errors="coerce")
pm_markets["start_date"] = pd.to_datetime(pm_markets["start_date"], utc=True, errors="coerce")
pm_markets["volume"] = pd.to_numeric(pm_markets["volume"], errors="coerce")
pm_markets["liquidity"] = pd.to_numeric(pm_markets["liquidity"], errors="coerce")

# Derive actual date range from trade data instead of API metadata
pm_market_dates = pm_trades.groupby("market_id")["timestamp"].agg(["min", "max"]).reset_index()
pm_market_dates.columns = ["market_id", "trade_start", "trade_end"]
pm_markets = pm_markets.merge(pm_market_dates, on="market_id", how="left")

print("Trade-derived date ranges:")
print(pm_markets[["market_id", "trade_start", "trade_end"]].head(5))
print(f"\nTrade end range: {pm_markets['trade_end'].min()} to {pm_markets['trade_end'].max()}")
print(f"Trades timestamp range: {pm_trades['timestamp'].min()} to {pm_trades['timestamp'].max()}")

Trade-derived date ranges:
                                           market_id  \
0  0xe3b423dfad8c22ff75c9899c4e8176f628cf4ad4caa0...   
1  0x44f10d1cd5aaed4b7ae0b5edb76790f54f45dc0bcaa8...   
2  0x3e0524de013d9dc359f5eb370773f25de2f03d320029...   
3  0x9b946f54f3428aafc308c33aa04a943fe13a011bdac9...   
4  0x5d1a1ab716fd06943441fe27cde0089651ce769bec55...   

                trade_start                 trade_end  
0 2026-06-01 01:35:11+00:00 2026-06-01 01:40:26+00:00  
1 2026-06-01 01:35:11+00:00 2026-06-01 01:40:26+00:00  
2 2026-06-01 01:35:11+00:00 2026-06-01 01:40:26+00:00  
3 2026-06-01 01:35:11+00:00 2026-06-01 01:40:26+00:00  
4 2026-06-01 01:35:11+00:00 2026-06-01 01:40:26+00:00  

Trade end range: 2026-06-01 01:40:26+00:00 to 2026-06-01 01:45:27+00:00
Trades timestamp range: 2026-06-01 01:35:11+00:00 to 2026-06-01 01:45:27+00:00


In [18]:
# Cell 4 — Clean and parse Kalshi trades
kalshi_trades["created_time"] = pd.to_datetime(kalshi_trades["created_time"], utc=True, errors="coerce")
kalshi_trades["yes_price_dollars"] = pd.to_numeric(kalshi_trades["yes_price_dollars"], errors="coerce")
kalshi_trades["count_fp"] = pd.to_numeric(kalshi_trades["count_fp"], errors="coerce")

kalshi_markets["end_date"] = pd.to_datetime(kalshi_markets["end_date"], utc=True, errors="coerce")
kalshi_markets["start_date"] = pd.to_datetime(kalshi_markets["start_date"], utc=True, errors="coerce")
kalshi_markets["volume"] = pd.to_numeric(kalshi_markets["volume"], errors="coerce")

print("Kalshi date range:")
print(f"  Earliest trade: {kalshi_trades['created_time'].min()}")
print(f"  Latest trade:   {kalshi_trades['created_time'].max()}")

Kalshi date range:
  Earliest trade: 2026-05-30 21:24:26.937814+00:00
  Latest trade:   2026-05-30 23:29:59.335844+00:00


In [19]:
# Cell 4b — Diagnose Polymarket trade/market ID mismatch
unique_condition_ids = pm_trades["conditionId"].unique()
print(f"Unique conditionIds in trades: {len(unique_condition_ids)}")
print(f"Unique market_ids in trades: {pm_trades['market_id'].nunique()}")

# Check if our market IDs match any conditionIds
pm_markets["has_trades"] = pm_markets["market_id"].isin(unique_condition_ids)
print(f"Markets matching conditionIds: {pm_markets['has_trades'].sum()}")

# Look at the slug fields — these may be the correct filter parameter
print("\nSample from trades:")
print(pm_trades[["conditionId", "title", "slug", "eventSlug"]].drop_duplicates("conditionId").head(10))

Unique conditionIds in trades: 579
Unique market_ids in trades: 99
Markets matching conditionIds: 0

Sample from trades:
                                          conditionId  \
0   0x8370c9f20f0c94acdcf42a595a6daec2e25efc60ed54...   
2   0xa87031a0326f50403887aa0f9e15574532d54271cd4c...   
4   0xab5bccf488b3653fc838711890aef67b38ad5e974b64...   
5   0x8f93b4c9e346fa79b1b2cca50825c0993527c2d971d1...   
6   0xc60f0c4582efa0c0b20dbb910feffbc73399c9c85f04...   
7   0xb8aae7863ca00c54c596a658c9908938c3e13b33c6e9...   
9   0xfd10096e2581115abe9a48ea4507686b2053374ee502...   
10  0x71dbeabf493e4baa6a817014288dbfb6595e479e41a2...   
11  0x69eafb44c424a6d082f68812ec3e6ea86687098d8fc7...   
12  0xd9fb1184af0064e5e34b129f5b79afa5a17b7e32f295...   

                                                title  \
0       Bitcoin Up or Down - May 31, 9:40PM-9:45PM ET   
2       Bitcoin Up or Down - May 31, 9:30PM-9:45PM ET   
4   Will the highest temperature in Hong Kong be 3...   
5   Will the highest te

In [13]:
# Cell 5 — Stage 1 features (market structure, one row per market)
def compute_stage1_features(markets, trades, timestamp_col, price_col, size_col,
                             start_col="trade_start", end_col="trade_end"):
    features = []

    for _, market in tqdm(markets.iterrows(), total=len(markets), desc="Stage 1 features"):
        market_id = market["market_id"]
        market_trades = trades[trades["market_id"] == market_id].copy()

        # Use trade-derived dates
        trade_end = market.get(end_col)
        trade_start = market.get(start_col)

        # Time to resolution in hours — derived from actual trade activity
        if pd.notna(trade_end) and pd.notna(trade_start):
            time_to_resolution_hrs = (trade_end - trade_start).total_seconds() / 3600
        else:
            time_to_resolution_hrs = np.nan

        # Opening probability — price of first trade
        if len(market_trades) > 0:
            market_trades_sorted = market_trades.sort_values(timestamp_col)
            opening_price = market_trades_sorted[price_col].iloc[0]
            uncertainty = 1 - abs(opening_price - 0.5) * 2
        else:
            opening_price = np.nan
            uncertainty = np.nan

        # Price volatility — std dev of price over first 10% of trades
        n_early = max(1, int(len(market_trades) * 0.1))
        early_trades = market_trades.sort_values(timestamp_col).head(n_early)
        price_volatility = early_trades[price_col].std() if len(early_trades) > 1 else np.nan

        # Trade count
        trade_count = len(market_trades)

        features.append({
            "market_id": market_id,
            "question": market.get("question"),
            "category": market.get("category"),
            "platform": market.get("platform"),
            "volume": market.get("volume"),
            "liquidity": market.get("liquidity"),
            "time_to_resolution_hrs": time_to_resolution_hrs,
            "opening_price": opening_price,
            "uncertainty": uncertainty,
            "price_volatility": price_volatility,
            "trade_count": trade_count,
            "outcome": market.get("outcome"),
            "trade_start": trade_start,
            "trade_end": trade_end,
        })

    return pd.DataFrame(features)

# Compute for Polymarket — uses trade-derived dates
pm_stage1 = compute_stage1_features(
    pm_markets, pm_trades,
    timestamp_col="timestamp",
    price_col="price",
    size_col="size",
    start_col="trade_start",
    end_col="trade_end"
)

# Compute for Kalshi — derive dates from trades too
kalshi_market_dates = kalshi_trades.groupby("market_id")["created_time"].agg(["min", "max"]).reset_index()
kalshi_market_dates.columns = ["market_id", "trade_start", "trade_end"]
kalshi_markets = kalshi_markets.merge(kalshi_market_dates, on="market_id", how="left")

kalshi_stage1 = compute_stage1_features(
    kalshi_markets, kalshi_trades,
    timestamp_col="created_time",
    price_col="yes_price_dollars",
    size_col="count_fp",
    start_col="trade_start",
    end_col="trade_end"
)

# Combine
stage1 = pd.concat([pm_stage1, kalshi_stage1], ignore_index=True)

# Convert numeric columns
for col in ["volume", "liquidity", "time_to_resolution_hrs", "opening_price",
            "uncertainty", "price_volatility", "trade_count"]:
    stage1[col] = pd.to_numeric(stage1[col], errors="coerce")

stage1.to_parquet("../data/processed/features_stage1.parquet", index=False)
print(f"\nStage 1 features: {len(stage1)} markets, {len(stage1.columns)} features")
stage1.head()

Stage 1 features: 100%|██████████| 13/13 [00:00<00:00, 1394.53it/s]


Stage 1 features: 113 markets, 14 features


,market_id,question,category,platform,volume,liquidity,time_to_resolution_hrs,opening_price,uncertainty,price_volatility,trade_count,outcome,trade_start,trade_end
0,0xe3b423dfad8c22ff75c9899c4e8176f628cf4ad4caa0...,Will Joe Biden get Coronavirus before the elec...,US-current-affairs,polymarket,32257.445115,0.000000,0.0875,0.99,0.02,0.232615,3500,None,2026-06-01 01:35:11+00:00,2026-06-01 01:40:26+00:00
1,0x44f10d1cd5aaed4b7ae0b5edb76790f54f45dc0bcaa8...,Will Airbnb begin publicly trading before Jan ...,Tech,polymarket,89665.252158,0.000000,0.0875,0.99,0.02,0.232615,3500,None,2026-06-01 01:35:11+00:00,2026-06-01 01:40:26+00:00
2,0x3e0524de013d9dc359f5eb370773f25de2f03d320029...,Will a new Supreme Court Justice be confirmed ...,US-current-affairs,polymarket,43279.456005,0.000000,0.0875,0.99,0.02,0.232615,3500,None,2026-06-01 01:35:11+00:00,2026-06-01 01:40:26+00:00
3,0x9b946f54f3428aafc308c33aa04a943fe13a011bdac9...,Will Kim Kardashian and Kanye West divorce bef...,Pop-Culture,polymarket,22067.475119,0.179651,0.0875,0.99,0.02,0.232615,3500,None,2026-06-01 01:35:11+00:00,2026-06-01 01:40:26+00:00
4,0x5d1a1ab716fd06943441fe27cde0089651ce769bec55...,Will Coinbase begin publicly trading before Ja...,Crypto,polymarket,116803.377183,0.367501,0.0875,0.99,0.02,0.232615,3500,None,2026-06-01 01:35:11+00:00,2026-06-01 01:40:26+00:00


In [14]:
# Cell 6 — Stage 2 features (behavioral, 48h pre-resolution window)
def compute_stage2_features(markets, trades, timestamp_col, price_col, size_col, side_col):
    features = []
    window_hrs = 48

    for _, market in tqdm(markets.iterrows(), total=len(markets), desc="Stage 2 features"):
        market_id = market["market_id"]
        end_date = market["end_date"]

        if pd.isna(end_date):
            continue

        market_trades = trades[trades["market_id"] == market_id].copy()
        if len(market_trades) < 10:
            continue

        market_trades = market_trades.sort_values(timestamp_col)

        # Define windows
        window_start = end_date - pd.Timedelta(hours=window_hrs)
        pre_window = market_trades[
            (market_trades[timestamp_col] >= window_start) &
            (market_trades[timestamp_col] <= end_date)
        ]
        baseline = market_trades[market_trades[timestamp_col] < window_start]

        if len(pre_window) < 3:
            continue

        # Volume z-score vs baseline
        baseline_vol = baseline[size_col].sum()
        baseline_hrs = max(1, (baseline[timestamp_col].max() - baseline[timestamp_col].min()).total_seconds() / 3600)
        baseline_rate = baseline_vol / baseline_hrs

        pre_vol = pre_window[size_col].sum()
        pre_rate = pre_vol / window_hrs

        vol_zscore = (pre_rate - baseline_rate) / (baseline[size_col].std() + 1e-9) if len(baseline) > 1 else 0

        # Order flow imbalance (BUY vs SELL pressure)
        if side_col in pre_window.columns:
            buys = pre_window[pre_window[side_col].str.upper() == "YES"][size_col].sum()
            sells = pre_window[pre_window[side_col].str.upper() == "NO"][size_col].sum()
            total = buys + sells
            order_flow_imbalance = (buys - sells) / (total + 1e-9)
        else:
            order_flow_imbalance = np.nan

        # Price drift velocity — slope of price over window
        if len(pre_window) > 1:
            prices = pre_window[price_col].values
            times = np.arange(len(prices))
            drift_velocity = np.polyfit(times, prices, 1)[0]
        else:
            drift_velocity = 0

        # Price impact — avg price change per unit volume
        price_changes = pre_window[price_col].diff().abs()
        size_values = pre_window[size_col]
        price_impact = (price_changes / (size_values + 1e-9)).mean()

        # Trade clustering — ratio of trades in final 6h vs rest of window
        final_6h_start = end_date - pd.Timedelta(hours=6)
        final_6h_trades = pre_window[pre_window[timestamp_col] >= final_6h_start]
        earlier_trades = pre_window[pre_window[timestamp_col] < final_6h_start]
        clustering_ratio = len(final_6h_trades) / (len(earlier_trades) + 1e-9)

        # Combined signal — volume z-score * abs(order flow imbalance)
        combined_signal = vol_zscore * abs(order_flow_imbalance) if not np.isnan(order_flow_imbalance) else vol_zscore

        features.append({
            "market_id": market_id,
            "platform": market.get("platform"),
            "end_date": end_date,
            "pre_window_trades": len(pre_window),
            "vol_zscore": vol_zscore,
            "order_flow_imbalance": order_flow_imbalance,
            "drift_velocity": drift_velocity,
            "price_impact": price_impact,
            "clustering_ratio": clustering_ratio,
            "combined_signal": combined_signal,
            "pre_window_volume": pre_vol,
            "outcome": market.get("outcome"),
        })

    return pd.DataFrame(features)

pm_stage2 = compute_stage2_features(
    pm_markets, pm_trades,
    timestamp_col="timestamp",
    price_col="price",
    size_col="size",
    side_col="side"
)

kalshi_stage2 = compute_stage2_features(
    kalshi_markets, kalshi_trades,
    timestamp_col="created_time",
    price_col="yes_price_dollars",
    size_col="count_fp",
    side_col="taker_side"
)

stage2 = pd.concat([pm_stage2, kalshi_stage2], ignore_index=True)
stage2.to_parquet("../data/processed/features_stage2.parquet", index=False)
print(f"\nStage 2 features: {len(stage2)} markets, {len(stage2.columns)} features")
stage2.head()

Stage 2 features: 100%|██████████| 13/13 [00:00<00:00, 343.37it/s]


Stage 2 features: 11 markets, 12 features


,market_id,platform,end_date,pre_window_trades,vol_zscore,order_flow_imbalance,drift_velocity,price_impact,clustering_ratio,combined_signal,pre_window_volume,outcome
0,KXMLBTOTAL-26MAY301605SDWSH-14,kalshi,2026-05-30 22:54:49+00:00,14,0,0.124934,0.005319,0.134992,1.400000e+10,0.0,866.22,no
1,KXMLBTOTAL-26MAY301605SDWSH-13,kalshi,2026-05-30 22:54:48+00:00,124,0,0.176809,0.004535,0.012108,1.240000e+11,0.0,10746.84,yes
2,KXMLBSPREAD-26MAY301610MILHOU-HOU7,kalshi,2026-05-30 23:10:29+00:00,129,0,0.656946,0.005440,0.011239,1.290000e+11,0.0,12781.18,yes
3,KXMLBSPREAD-26MAY301610MILHOU-HOU6,kalshi,2026-05-30 23:10:29+00:00,90,0,0.709312,0.010652,0.017306,9.000000e+10,0.0,2169.82,yes
4,KXMLBSPREAD-26MAY301610MIANYM-NYM6,kalshi,2026-05-30 23:00:18+00:00,68,0,-0.196414,-0.005888,0.004305,6.800000e+10,0.0,3678.41,no


In [15]:
# Cell 7 — Sanity check
print("=== Feature Engineering Summary ===")
print(f"\nStage 1 ({len(stage1)} markets):")
print(stage1[["platform", "volume", "uncertainty", "time_to_resolution_hrs", "price_volatility"]].describe())

print(f"\nStage 2 ({len(stage2)} markets):")
print(stage2[["vol_zscore", "order_flow_imbalance", "drift_velocity", "clustering_ratio", "combined_signal"]].describe())

print("\nMissing values in Stage 1:")
print(stage1.isna().sum())

print("\nMissing values in Stage 2:")
print(stage2.isna().sum())

=== Feature Engineering Summary ===

Stage 1 (113 markets):
             volume  uncertainty  time_to_resolution_hrs  price_volatility
count  1.130000e+02   111.000000              111.000000        109.000000
mean   8.519022e+05     0.634423                0.174391          0.262258
std    3.223596e+06     0.289974                0.391847          0.056058
min    0.000000e+00     0.020000                0.031944          0.034859
25%    4.420485e+04     0.573737                0.031944          0.263345
50%    1.319441e+05     0.620000                0.086389          0.271358
75%    2.704060e+05     0.860000                0.087500          0.296675
max    3.020758e+07     0.980000                1.970004          0.319848

Stage 2 (11 markets):
       vol_zscore  order_flow_imbalance  drift_velocity  clustering_ratio  \
count        11.0             11.000000       11.000000      1.100000e+01   
mean          0.0              0.198243        0.001957      3.301818e+11   
std        

In [16]:
# Diagnose date issues
print("PM markets end_date sample:")
print(pm_markets[["market_id", "start_date", "end_date"]].head(10))

# Diagnose baseline issue
test_id = pm_trades["market_id"].iloc[0]
test_market = pm_markets[pm_markets["market_id"] == test_id].iloc[0]
test_trades = pm_trades[pm_trades["market_id"] == test_id].sort_values("timestamp")

print(f"\nTest market end_date: {test_market['end_date']}")
print(f"Test trades timestamp range:")
print(f"  First: {test_trades['timestamp'].min()}")
print(f"  Last:  {test_trades['timestamp'].max()}")
print(f"  Total trades: {len(test_trades)}")

# Check window
if pd.notna(test_market['end_date']):
    window_start = test_market['end_date'] - pd.Timedelta(hours=48)
    pre = test_trades[test_trades["timestamp"] >= window_start]
    baseline = test_trades[test_trades["timestamp"] < window_start]
    print(f"\n48h window start: {window_start}")
    print(f"Pre-window trades: {len(pre)}")
    print(f"Baseline trades: {len(baseline)}")

PM markets end_date sample:
                                           market_id  \
0  0xe3b423dfad8c22ff75c9899c4e8176f628cf4ad4caa0...   
1  0x44f10d1cd5aaed4b7ae0b5edb76790f54f45dc0bcaa8...   
2  0x3e0524de013d9dc359f5eb370773f25de2f03d320029...   
3  0x9b946f54f3428aafc308c33aa04a943fe13a011bdac9...   
4  0x5d1a1ab716fd06943441fe27cde0089651ce769bec55...   
5  0xd903891c2b9046cae14615afc4c5245370143503f7b2...   
6  0xf2e631ea675c5b09caea0bf65cf7887e25907af2657c...   
7  0xfa8cc293e14872fca0c7e8b240360683e392e1d7a4b5...   
8  0x7333b6e016f7f60d86f15f11ed0b41b69deec0b6d73b...   
9  0xb2eecb8d14e871c5b82a3b037fc5f8b703c218e41aa5...   

                        start_date                  end_date  
0                              NaT 2020-11-04 00:00:00+00:00  
1                              NaT 2021-01-02 00:00:00+00:00  
2                              NaT 2020-11-04 00:00:00+00:00  
3 2023-08-24 16:35:24.938000+00:00 2021-01-02 00:00:00+00:00  
4                              NaT 2021-

In [17]:
# Check how many unique market_ids are actually in the trades
print(f"Unique markets in pm_trades: {pm_trades['market_id'].nunique()}")
print(f"Unique conditionIds in pm_trades: {pm_trades['conditionId'].nunique()}")

# Check if conditionId matches market_id
print("\nSample market_id vs conditionId:")
print(pm_trades[["market_id", "conditionId", "title", "timestamp"]].head(10))

Unique markets in pm_trades: 99
Unique conditionIds in pm_trades: 579

Sample market_id vs conditionId:
                                           market_id  \
0  0xe3b423dfad8c22ff75c9899c4e8176f628cf4ad4caa0...   
1  0xe3b423dfad8c22ff75c9899c4e8176f628cf4ad4caa0...   
2  0xe3b423dfad8c22ff75c9899c4e8176f628cf4ad4caa0...   
3  0xe3b423dfad8c22ff75c9899c4e8176f628cf4ad4caa0...   
4  0xe3b423dfad8c22ff75c9899c4e8176f628cf4ad4caa0...   
5  0xe3b423dfad8c22ff75c9899c4e8176f628cf4ad4caa0...   
6  0xe3b423dfad8c22ff75c9899c4e8176f628cf4ad4caa0...   
7  0xe3b423dfad8c22ff75c9899c4e8176f628cf4ad4caa0...   
8  0xe3b423dfad8c22ff75c9899c4e8176f628cf4ad4caa0...   
9  0xe3b423dfad8c22ff75c9899c4e8176f628cf4ad4caa0...   

                                         conditionId  \
0  0x8370c9f20f0c94acdcf42a595a6daec2e25efc60ed54...   
1  0x8370c9f20f0c94acdcf42a595a6daec2e25efc60ed54...   
2  0xa87031a0326f50403887aa0f9e15574532d54271cd4c...   
3  0x8370c9f20f0c94acdcf42a595a6daec2e25efc60ed54...   